### Case of an Option which has bid=0 ask=0.01 and someone bought a lot of those calls

In [1]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.massive.client
%aimport quantdata.massive.processing
%aimport quantdata.massive.options.contracts

In [10]:
import polars as pl

from quantdata.biz_calendar import prev_business_day, to_date
from quantdata.massive.client import MassiveClient
from quantdata.pool import PoolConfig
from quantdata.massive.options.contracts import ContractsFilter, filter_contracts, REJECTED
from quantdata.massive.processing import align_trades_to_quotes, align_quotes_to_trades, filter_valid_trades

from cdxcore.version import version
from cdxcore.verbose import Context
from dataclasses import asdict, dataclass, field
from cdxcore.uniquehash import unique_hash8 
import numpy as np
import math

from quantdata.ncde.implied.day import _ncde_implied_day_root as _cache
from IPython.display import display, Markdown

from quantdata.massive.processing import filter_valid_quotes
from quantdata.massive.client import MassiveClient

opt_ticker = "O:SPY250123C00614000"
ticker = "SPY"
d = to_date("2025-01-23")
client = MassiveClient()
display(Markdown(f"### {opt_ticker} {d} has_any_quotes: {client.has_any_quotes(opt_ticker, d)}; has_any_trades: {client.has_any_trades(opt_ticker, d)}"))
display(Markdown(f"### fetch quotes {opt_ticker} on {d}"))
display( client.fetch_quotes(opt_ticker, d) )
display(Markdown(f"### fetch trades {opt_ticker} on {d}"))
display( client.fetch_trades(opt_ticker, d) )
display(Markdown(f"### valid quotes {opt_ticker} on {d}"))
display( filter_valid_quotes( opt_ticker, d) )
display(Markdown(f"### valid trades {opt_ticker} on {d}"))
display( filter_valid_trades( opt_ticker, d) )
display( align_trades_to_quotes( opt_ticker, d, within_session=True, drop_unmatched=False ) )


### O:SPY250123C00614000 2025-01-23 has_any_quotes: True; has_any_trades: True

### fetch quotes O:SPY250123C00614000 on 2025-01-23

ts_ns,bid_price,bid_size,ask_price,ask_size
"datetime[ns, America/New_York]",f64,f64,f64,f64
2025-01-23 09:30:00.003626496 EST,0.0,0.0,0.01,126.0
2025-01-23 11:03:22.858710528 EST,0.0,0.0,0.01,10145.0
2025-01-23 11:04:01.341701888 EST,0.0,0.0,0.01,3670.0
2025-01-23 11:11:40.969886976 EST,0.0,0.0,0.01,6671.0
2025-01-23 11:11:51.696362496 EST,0.0,0.0,0.01,8252.0
…,…,…,…,…
2025-01-23 13:50:27.365222400 EST,0.0,0.0,0.01,14009.0
2025-01-23 13:59:47.323405824 EST,0.0,0.0,0.01,15795.0
2025-01-23 14:02:02.649653504 EST,0.0,0.0,0.01,17129.0


### fetch trades O:SPY250123C00614000 on 2025-01-23

ts_ns,price,size,trade_code
"datetime[ns, America/New_York]",f64,f64,u32
2025-01-23 09:30:00.863 EST,0.01,1.0,256
2025-01-23 09:30:02.205 EST,0.01,2.0,256
2025-01-23 09:30:06.209 EST,0.01,1.0,256
2025-01-23 09:30:06.209 EST,0.01,10.0,256
2025-01-23 09:30:06.209 EST,0.01,25.0,256
…,…,…,…
2025-01-23 15:59:50.372 EST,0.01,2.0,2112
2025-01-23 15:59:50.372 EST,0.01,1.0,2112
2025-01-23 15:59:50.372 EST,0.01,1.0,2112


### valid quotes O:SPY250123C00614000 on 2025-01-23

ts_ns,bid_price,bid_size,ask_price,ask_size
"datetime[ns, America/New_York]",f64,f64,f64,f64


### valid trades O:SPY250123C00614000 on 2025-01-23

ts_ns,price,size,trade_code
"datetime[ns, America/New_York]",f64,f64,u32
2025-01-23 09:30:00.863 EST,0.01,1.0,256
2025-01-23 09:30:02.205 EST,0.01,2.0,256
2025-01-23 09:30:06.209 EST,0.01,1.0,256
2025-01-23 09:30:06.209 EST,0.01,10.0,256
2025-01-23 09:30:06.209 EST,0.01,25.0,256
…,…,…,…
2025-01-23 15:59:50.368 EST,0.01,3.0,2112
2025-01-23 15:59:50.372 EST,0.01,2.0,2112
2025-01-23 15:59:50.372 EST,0.01,2.0,2112


ts_ns,bid_price,bid_size,ask_price,ask_size,trade_ts_ns,price,size,trade_code
"datetime[ns, America/New_York]",f64,f64,f64,f64,"datetime[ns, America/New_York]",f64,f64,u32
